# V100 - Full build: bagged base plus no-chat action

Pre-registered plan: v98 bagging is the only adopted finding, so the full build replicates it under a fresh seed, bags the assessment view for the no-chat action base, re-tests the no-chat override on the stronger base, and writes a champion-relative submission only if every lock passes. OOF bagging for verdicts, full-train bagging for test predictions. Fail-close writes the untouched champion.

In [ ]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 7777
N_SPLITS = 5
GATE_FLOOR = 0.002
SLICE_FLOOR = 0.01
BAGS = 10
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "full build composes onto the champion file"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v100")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION)

## Replicate bagging, bag assessment, test the action

Single Ridge, bagged word, and bagged assessment share folds. The action overrides no-chat rows of the bagged word base with bagged assessment.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
W_tr, A_tr = W[:n_train], A[:n_train]
is_nochat = np.array([u not in by_user for u in all_users])
nochat_tr = is_nochat[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_oof(Xtr, seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[idx], Y[idx, j])
            acc[oob, j] += m.predict(Xtr[oob])
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

def single_oof(Xtr):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_s, fid = single_oof(W_tr)
oof_bw = bagged_oof(W_tr, SEED)
oof_ba = bagged_oof(A_tr, SEED + 1)
base = ndcg5(Y, oof_s)
d = ndcg5(Y, oof_bw) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_bw[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"single={base.mean():.5f} bagged-word={ndcg5(Y, oof_bw).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
rep_ok = (lo > GATE_FLOOR) and (wf >= 0)
print("REPLICATE CONFIRMED" if rep_ok else "REPLICATE FAILED: stop, keep champion")
act = oof_bw.copy()
act[nochat_tr] = oof_ba[nochat_tr]
da = ndcg5(Y, act) - ndcg5(Y, oof_bw)
ds = da[nochat_tr]
lo_s, hi_s = np.quantile([rng.choice(ds, len(ds), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf_a = min(ndcg5(Y[fid == k], act[fid == k]).mean() - ndcg5(Y, oof_bw)[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"action pooled={da.mean():+.5f} slice CI95=[{lo_s:+.5f},{hi_s:+.5f}] worst-fold={wf_a:+.5f}")
act_ok = rep_ok and (lo_s > SLICE_FLOOR) and (wf_a >= 0)
print("ACTION ADOPTED" if act_ok else "ACTION REJECTED")

## Test predictions and champion-relative submission

Full-train bagged ensembles score test. The file touches champion rows only inside the no-chat slice and only when every lock passed.

In [ ]:
def bagged_test(Xtr, Xte, seed):
    r = np.random.RandomState(seed)
    pred = np.zeros((len(Xte), len(MODS)))
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[idx], Y[idx, j])
            pred[:, j] += m.predict(Xte) / BAGS
    return pred

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
report = {"replicate": bool(rep_ok), "action": bool(act_ok)}
final = C.copy()
if act_ok:
    pW = bagged_test(W_tr, W[n_train:], SEED)
    pA = bagged_test(A_tr, A[n_train:], SEED + 1)
    S = C.copy()
    S[is_nochat[n_train:]] = pA[is_nochat[n_train:]]
    changed = set(np.where((S != C).any(1))[0].tolist())
    allowed = set(np.where(is_nochat[n_train:])[0].tolist())
    assert changed <= allowed, "geometry breach"
    t1 = int((C.argmax(1) != S.argmax(1)).sum())
    report.update({"changed": len(changed), "top1_changes": t1})
    print(f"geometry: changed={len(changed)} all-in-slice=True top1-changes={t1}")
    final = S
else:
    print("FAIL-CLOSE: champion untouched")
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v100_primary.csv", index=False)
report["sha"] = sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16]
report["champion_sha"] = sha256(pd.util.hash_pandas_object(ch).values.tobytes()).hexdigest()[:16]
Path(OUT / "v100_gate_report.json").write_text(json.dumps(report, indent=1))
print("wrote", OUT / "submission_v100_primary.csv", report)